In [2]:
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler
# from lora_diffusion.lora import tune_lora_scale, patch_pipe # <-- 不再需要，删除或注释掉
import torch

# 1. 加载基础模型和调度器 (这部分保持不变)
model_id = "/root/stable-diffusion-2-1-base"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to(
    "cuda:6"
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

# 2. 定义 LoRA 路径
lora_path = "/root/lora_train/multi_combine_20250726_131443/multi_teacher_distilled/final_multi_teacher_hybrid_lora_step_5000.safetensors"

# 3. 【修改】使用官方方法加载 LoRA
# 这一行替代了原来的 patch_pipe(...)
pipe.load_lora_weights(lora_path)

# 4. 定义 prompt (保持不变)
prompt = "a photo of a great <red_light_diode_front>"

# 5. 【修改】设置 LoRA 权重 (替代 tune_lora_scale)
# 在官方方法中，通过 cross_attention_kwargs 字典来控制 LoRA 强度
# 1.0 是默认值，但这里显式写出来以对应您原始代码的逻辑
lora_scale = 1.0

# 6. 运行推理 (在 pipe 调用时传入 lora_scale)
torch.manual_seed(42)
image = pipe(
    prompt,
    num_inference_steps=50,
    guidance_scale=7.5,
    cross_attention_kwargs={"scale": lora_scale}  # <-- 在这里应用 LoRA 权重
).images[0]

# 7. 保存图像 (保持不变)
image.save("../contents/lion_illust.jpg")
image

Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00,  9.12it/s]


ValueError: PEFT backend is required for this method.